Semana 3 Dia 1
Test Binomial exacto

In [4]:
from scipy.stats import binomtest

# 14 éxitos de 20 intentos, bajo H0: p=0.5
resultado = binomtest(14, n=20, p=0.5, alternative='greater')
print(f"p-value: {resultado.pvalue:.4f}")

if resultado.pvalue < 0.05:
    print("→ Rechazamos H0: evidencia de que el 70% NO es pura suerte")
else:
    print("→ NO rechazamos H0: no hay evidencia suficiente, podría ser suerte")

p-value: 0.0577
→ NO rechazamos H0: no hay evidencia suficiente, podría ser suerte


In [5]:
resultado_40 = binomtest(28, n=40, p=0.5, alternative='greater')
print(f"p-value con 40 trades (mismo 70%): {resultado_40.pvalue:.4f}")

p-value con 40 trades (mismo 70%): 0.0083


In [7]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm

# Mercado proxy y activo a analizar
tickers = ["SPY", "TSLA"]
data = yf.download(tickers, start="2020-01-01", end="2024-12-31")["Close"]
returns = np.log(data / data.shift(1)).dropna()

r_mercado = returns["SPY"]
r_activo = returns["TSLA"]

# Regresión: r_activo = alpha + beta * r_mercado + epsilon
X = sm.add_constant(r_mercado)  # agrega el intercepto (alpha)
modelo = sm.OLS(r_activo, X).fit()

print(modelo.summary())

[*********************100%***********************]  2 of 2 completed

                            OLS Regression Results                            
Dep. Variable:                   TSLA   R-squared:                       0.262
Model:                            OLS   Adj. R-squared:                  0.261
Method:                 Least Squares   F-statistic:                     444.9
Date:                Sun, 21 Jun 2026   Prob (F-statistic):           9.17e-85
Time:                        18:42:12   Log-Likelihood:                 2384.3
No. Observations:                1256   AIC:                            -4765.
Df Residuals:                    1254   BIC:                            -4754.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0013      0.001      1.235      0.2

In [12]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from scipy.stats import binomtest

# --- CAPM para varios activos ---
tickers_analisis = ["TSLA", "AAPL", "GLD"]
tickers_completos = ["SPY"] + tickers_analisis

data = yf.download(tickers_completos, start="2025-01-01", end="2026-06-21")["Close"]
returns = np.log(data / data.shift(1)).dropna()

r_mercado = returns["SPY"]

print("CAPM — Beta, Alpha y R² por activo:")
print("=" * 70)

for ticker in tickers_analisis:
    r_activo = returns[ticker]
    X = sm.add_constant(r_mercado)
    modelo = sm.OLS(r_activo, X).fit()
    
    beta = modelo.params["SPY"]
    alpha = modelo.params["const"]
    alpha_pvalue = modelo.pvalues["const"]
    r2 = modelo.rsquared
    
    alpha_anualizado = alpha * 252
    
    print(f"\n{ticker}:")
    print(f"  Beta:              {beta:.3f}")
    print(f"  Alpha diario:      {alpha:.5f}  (anualizado: {alpha_anualizado*100:.2f}%)")
    print(f"  Alpha p-value:     {alpha_pvalue:.4f}  → "
          f"{'Significativo' if alpha_pvalue < 0.05 else 'NO significativo'}")
    print(f"  R²:                {r2:.3f}  "
          f"({r2*100:.1f}% explicado por el mercado)")

[*********************100%***********************]  4 of 4 completed

CAPM — Beta, Alpha y R² por activo:

TSLA:
  Beta:              2.128
  Alpha diario:      -0.00138  (anualizado: -34.69%)
  Alpha p-value:     0.3245  → NO significativo
  R²:                0.448  (44.8% explicado por el mercado)

AAPL:
  Beta:              1.169
  Alpha diario:      -0.00027  (anualizado: -6.83%)
  Alpha p-value:     0.7019  → NO significativo
  R²:                0.488  (48.8% explicado por el mercado)

GLD:
  Beta:              0.213
  Alpha diario:      0.00110  (anualizado: 27.62%)
  Alpha p-value:     0.1925  → NO significativo
  R²:                0.022  (2.2% explicado por el mercado)
